Refer: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md

# Setup
Library `gitsource` downloads files from a GitHub repository.

In [57]:
! uv add gitsource

Resolved 164 packages in 2ms
Audited 160 packages in 17ms


# Preparation
`GithubRepositoryDataReader` downloads the entire repository and goes over all the files in it. `allowed_extensions={"md"}` checks only the markdown files.

We also pass a `filename_filter` so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's `lessons/` folder, so filtering on `/lessons/` keeps just those.

Each file has a `parse()` method that returns a dictionary with its `filename` and `content`:

In [58]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

How many lesson pages are in the dataset?
* 24
* `72`
* 240
* 720

In [59]:
print(f"Number of documents: {len(documents)}")

Number of documents: 72


## Q2. Indexing and searching
Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:

`How does the agentic loop keep calling the model until it stops?`

What's the `filename` of the first result?

* 01-agentic-rag/lessons/03-rag.md
* `01-agentic-rag/lessons/14-agentic-loop.md`
* 04-evaluation/lessons/13-llm-as-judge.md
* 06-best-practices/lessons/02-hybrid-search.md

In [60]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [61]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.

Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?
* 700
* `7000`
* 70000
* 700000

We count input tokens instead of price because the cost depends on the model
and provider you use, but the size of the prompt we send is the same for
everyone.

Most LLM APIs report token usage on the response object (e.g.
`response.usage.input_tokens` / `prompt_tokens`). We'll read the input tokens
from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, `llm` returns only the text. Modify it to return the whole response, and change `rag` to return both the answer and usage (as a tuple or create a small dataclass for that).

In [62]:
from modified_rag_helper import DocumentRAG
from dotenv import load_dotenv
load_dotenv()
import os
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [63]:
# Initialize custom RAG
assistant = DocumentRAG(index=index, llm_client=client)

In [64]:
query='How does the agentic loop keep calling the model until it stops?'
answer, usage = assistant.rag(query)



In [65]:
print(f"Input tokens: {usage['input_tokens']}")

Input tokens: 7183


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

* 70
* `295`
* 1100
* 4500

In [66]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [67]:
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?

* about the same
* 3× fewer
* `10× fewer`
* 30× fewer

In [68]:
from ingest import build_index
chunk_index = build_index(chunks)
rag_chunked = DocumentRAG(index=chunk_index, llm_client=client)
answer, usage = rag_chunked.rag(query)

In [69]:
print(f"Input tokens: {usage['input_tokens']}")

Input tokens: 109



## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.

If you go with toyaikit:

```bash
uv add toyaikit
```

Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

> Note: the agent decides this itself, so it varies a little between runs -
> pick the closest option. We measured this with OpenAI `gpt-5.4-mini`; with a
> different model or provider the number may differ, so keep that in mind.

* 0
* 4
* `10`
* 20


In [70]:
! uv add langchain langchain-openai langchain-community langgraph

Resolved 164 packages in 1ms
Audited 160 packages in 14ms


In [71]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage
import os
import json

### Convert search to a LangChain Tool:
LangChain needs tools in a specific format. You can use the `@tool` decorator to convert your function

In [72]:
@tool
def search(query: str) -> str:
    """
    Search the course lesson chunks for information matching the given query.
    Use this tool to find relevant content from the course materials.
    """
    boost_dict = {"content": 3.0, "filename": 0.5}
    
    results = chunk_index.search(
        query=query,
        num_results=5,
        boost_dict=boost_dict
    )
    
    if not results:
        return "No results found."
    
    formatted_results = []
    for i, result in enumerate(results, 1):
        formatted_results.append(f"Result {i} (from {result.get('filename', 'unknown')}):")
        content = result.get('content', '')
        formatted_results.append(content[:800] + "..." if len(content) > 800 else content)
        formatted_results.append("-" * 50)
    
    return "\n".join(formatted_results)

### Set Up the Agent using LangGraph:


In [73]:
# Initialize the LLM
llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [74]:
# Create tools list
tools = [search]


In [75]:
# Create the agent with proper configuration
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""You are a course teaching assistant. You have access to a search tool.

        Guidelines:
        1. Use the search tool to find information from the course lessons
        2. Make multiple searches with different keywords to get comprehensive information
        3. Synthesize information from multiple search results
        4. If you don't find relevant information, try different search terms
        5. Provide a complete, well-structured answer with examples if possible

        When you have enough information, provide your final answer directly to the user.
        """
)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_12860\542822115.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
question = "How does the agentic loop work, and how is it different from plain RAG?"
# Run the agent
result = agent.invoke(
        {"messages": [("user", question)]}
)

In [80]:
final_message = result["messages"][-1]
print("\n" + "="*60)
print("FINAL ANSWER:")
print("="*60)
print(final_message.content)
print("="*60)


FINAL ANSWER:
**The agentic loop – what it is and why it is more than “plain” RAG**

| Aspect | Plain Retrieval‑Augmented Generation (RAG) | Agentic Loop (LLM‑as‑agent) |
|--------|--------------------------------------------|-----------------------------|
| **Control flow** | One‑shot: <br> 1️⃣ User query → 2️⃣ Retriever returns a fixed set of documents → 3️⃣ LLM consumes the docs and emits a response. | Iterative, self‑directed: <br> 1️⃣ Perceive (read the query) → 2️⃣ Reason about what to do → 3️⃣ Choose a tool (retriever, calculator, web‑search, database, etc.) → 4️⃣ Execute the tool → 5️⃣ Observe the result → 6️⃣ Update internal state/memory → 7️⃣ Repeat until a stopping condition (answer ready, max steps, confidence threshold). |
| **Decision making** | The pipeline is hard‑wired; the LLM never decides *whether* to retrieve or *which* tool to use. | The LLM **acts as a planner**: it can decide “I need more information → call the retriever”, “I need to run a calculation → call a 

In [79]:
# Count tool calls
tool_calls = 0
for message in result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        for tool_call in message.tool_calls:
                # Skip the 'commentary' tool if it exists
            if tool_call.get("name") == "search":
                tool_calls += 1
                print(f" Search #{tool_calls}: {tool_call.get('args', {}).get('query', '')}")
    
print(f"\nThe search tool was called {tool_calls} times.")


 Search #1: agentic loop
 Search #2: agentic
 Search #3: RAG
 Search #4: agentic loop RAG
 Search #5: "agentic loop"
 Search #6: agentic loop AI
 Search #7: retrieval augmented generation
 Search #8: agentic

The search tool was called 8 times.
